# Experimento GRPO de traducción: recorrido guiado y rápido

Este notebook explica y ejecuta, por etapas, el experimento `ours` que normalmente se inicia con `uv run python scripts/run_all.py --models 600M --langs rus_Cyrl --methods ours`.

La configuración inicial está pensada para **comprobar el flujo en poco tiempo**, no para obtener resultados científicos: usa pocos ejemplos, una época, LoRA, ejecuta test para que el recolector pueda mostrar chrF++ y no guarda el modelo. El entrenamiento solo empieza si cambias explícitamente `RUN_TRAINING` a `True` en la celda de ajustes.

LoRA reduce los parámetros entrenables y el coste asociado a sus gradientes/actualizaciones, pero el modelo base sigue generando traducciones. Por eso, para acelerar mucho una prueba también limitamos los datos de entrenamiento y evaluación y desactivamos las evaluaciones intermedias.


## Mapa del proceso

1. Encontrar la raíz del repositorio y comprobar que existe el entorno `uv` del proyecto.
2. Elegir modelo, idioma, LoRA, límites de datos, épocas y política de evaluación.
3. (Opcional) Consultar los tamaños reales de los splits y estimar lotes.
4. Construir y revisar los overrides de Hydra. Corresponden a los argumentos que el script de ejecución prepara para `main.py`.
5. Lanzar el entrenamiento de forma explícita.

El detalle de carga del modelo, preparación de datos, evaluación, bucle GRPO y guardado está implementado en `main.py`; este notebook lo reutiliza en lugar de mantener una segunda copia del algoritmo que podría divergir.

### Qué hace `main.py` cuando se ejecuta el último paso

1. Carga el tokenizer y NLLB 600M; con LoRA activado, congela el modelo base y adapta las capas `q_proj` y `v_proj`.
2. Carga el dataset `breakend/nllb-multi-domain` para el par inglés-ruso y aplica los límites seleccionados. El entrenamiento usa lotes de `BATCH_SIZE`; validación y test usan lotes de 48.
3. Prepara una copia congelada como política de referencia y el optimizador con los parámetros entrenables.
4. Si están activadas, calcula las métricas iniciales de validación y test.
5. Por cada lote de entrenamiento, genera candidatos inglés→ruso y ruso→inglés, calcula la recompensa chrF, calcula la pérdida GRPO, hace backpropagation y actualiza el optimizador.
6. Si `EVAL_EVERY_N_OPT_STEPS > 0`, vuelve a evaluar validación en los pasos indicados. Después de entrenar, calcula las métricas finales que estén activadas y, si se pidió, guarda el modelo.

Las celdas anteriores separan configuración, inspección y estimación; el bucle de entrenamiento real se mantiene en `main.py` y se inicia desde la última celda para conservar el mismo algoritmo que usa el comando original.

In [1]:
from pathlib import Path
import os
import shlex
import subprocess
import sys
from datetime import datetime

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (path.resolve() for path in candidates if (path / "main.py").is_file() and (path / "configs" / "train.yaml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("No encuentro la raíz del repositorio (se esperaba main.py y configs/train.yaml en un mismo directorio).")

print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Python del kernel: {sys.executable}")
print(f"uv disponible: {subprocess.run(['uv', '--version'], cwd=PROJECT_ROOT, check=True, text=True, capture_output=True).stdout.strip()}")

Raíz del proyecto: /home/miguel/MT-via-Round-Trip-RL
Python del kernel: /home/miguel/MT-via-Round-Trip-RL/.venv/bin/python
uv disponible: uv 0.13.1 (x86_64-unknown-linux-gnu)


## 1. Entorno de ejecución

El entrenamiento se ejecuta mediante `uv run` desde la raíz del proyecto, así que usa las dependencias declaradas en `pyproject.toml` aunque el kernel de Jupyter tenga otro intérprete. Esta comprobación no descarga el modelo ni inicia el entrenamiento.

In [2]:
dependency_check = subprocess.run(
    ["uv", "run", "python", "-c", "import torch, transformers, datasets, hydra, omegaconf; print('torch', torch.__version__); print('transformers', transformers.__version__); print('datasets', datasets.__version__)"],
    cwd=PROJECT_ROOT,
    check=True,
    text=True,
    capture_output=True,
)
print(dependency_check.stdout)

torch 2.8.0+cu128
transformers 4.57.1
datasets 3.6.0



## 2. Ajustes del experimento

Edita esta celda antes de ejecutar las siguientes. `MAX_TRAIN_SAMPLES` y `MAX_EVAL_SAMPLES` son límites: `None` significa usar el split completo. El límite de evaluación se aplica tanto a validación como a test. `EVAL_EVERY_N_OPT_STEPS = 0` desactiva las evaluaciones **intermedias**, pero si `RUN_VALIDATION` está activo se conserva una evaluación inicial y otra final. `RUN_TEST` controla aparte las evaluaciones de test inicial y final.

La configuración actual amplía la prueba previa: 16 ejemplos de entrenamiento durante 2 épocas (16 pasos), 32 ejemplos de validación/test y LoRA. El tope de ejecución es 14 minutos para dejar margen bajo el límite de 15. Sigue siendo una muestra pequeña, no un resultado científico. `RUN_TRAINING` sigue en `False` para evitar lanzarlo por accidente; actívalo explícitamente si quieres repetirla. Cada ejecución se guardará en una carpeta nueva dentro de `results/notebook_runs/`.

In [3]:
MODEL_KEY = "600M"
LANG = "rus_Cyrl"
USE_LORA = True

MAX_TRAIN_SAMPLES = 16  # None para usar los 6.000 ejemplos de entrenamiento
MAX_EVAL_SAMPLES = 32   # None para usar los splits completos de validación y test
EPOCHS = 2
BATCH_SIZE = 2
EVAL_EVERY_N_OPT_STEPS = 0  # 0 desactiva la evaluación intermedia
RUN_VALIDATION = True
RUN_TEST = True  # necesario para obtener las columnas base/final de collect_results.py
USE_WANDB = False
SAVE_MODEL = False

# Interruptores de seguridad: no inspeccionan datos ni entrenan hasta que los habilites.
INSPECT_DATASET = False
RUN_TRAINING = False

MODELS = {
    "600M": "facebook/nllb-200-distilled-600M",
    "1.3B": "facebook/nllb-200-distilled-1.3B",
}
if MODEL_KEY not in MODELS:
    raise ValueError(f"MODEL_KEY debe ser una de estas opciones: {list(MODELS)}")
if EPOCHS < 1 or BATCH_SIZE < 1:
    raise ValueError("EPOCHS y BATCH_SIZE deben ser enteros positivos.")
if EVAL_EVERY_N_OPT_STEPS < 0:
    raise ValueError("EVAL_EVERY_N_OPT_STEPS no puede ser negativo.")
for name, limit in (("MAX_TRAIN_SAMPLES", MAX_TRAIN_SAMPLES), ("MAX_EVAL_SAMPLES", MAX_EVAL_SAMPLES)):
    if limit is not None and limit < 1:
        raise ValueError(f"{name} debe ser None o un entero positivo.")

print(f"Modelo: {MODELS[MODEL_KEY]} | idioma destino: {LANG}")
print(f"LoRA: {USE_LORA} | train: {MAX_TRAIN_SAMPLES} | eval/test: {MAX_EVAL_SAMPLES}")
print(f"Épocas: {EPOCHS} | batch: {BATCH_SIZE} | evaluación intermedia cada: {EVAL_EVERY_N_OPT_STEPS} pasos")

Modelo: facebook/nllb-200-distilled-600M | idioma destino: rus_Cyrl
LoRA: True | train: 16 | eval/test: 32
Épocas: 2 | batch: 2 | evaluación intermedia cada: 0 pasos


## 3. (Opcional) Inspeccionar los datos y contar lotes

Activa `INSPECT_DATASET` y ejecuta esta celda para cargar el dataset de Hugging Face y contar sus splits. La primera carga puede tardar o necesitar conexión a internet. Para una muestra limitada, los lotes de entrenamiento se calculan como `ceil(min(tamaño_train, MAX_TRAIN_SAMPLES) / BATCH_SIZE)`; validación y test usan un batch de 48, como en `dl.py`.

El cálculo de evaluaciones intermedias es una estimación: supone un paso de optimización por lote, como ocurre con los ajustes de este notebook.

In [4]:
if not INSPECT_DATASET:
    print("Inspección omitida. Cambia INSPECT_DATASET a True para cargar y contar los splits.")
else:
    import json
    dataset_config = f"eng_Latn-{LANG}"
    dataset_code = (
        "from datasets import load_dataset; import json; "
        f"data = load_dataset('breakend/nllb-multi-domain', '{dataset_config}', trust_remote_code=True); "
        "print(json.dumps({name: len(split) for name, split in data.items()}))"
    )
    result = subprocess.run(
        ["uv", "run", "python", "-c", dataset_code],
        cwd=PROJECT_ROOT,
        check=True,
        text=True,
        capture_output=True,
    )
    split_sizes = json.loads(result.stdout.strip().splitlines()[-1])
    train_size = split_sizes.get("train", split_sizes.get("training", 0))
    eval_size = split_sizes.get("valid", split_sizes.get("validation", split_sizes.get("val", 0)))
    test_size = split_sizes.get("test", split_sizes.get("test_final", 0))
    effective_train = min(train_size, MAX_TRAIN_SAMPLES) if MAX_TRAIN_SAMPLES is not None else train_size
    effective_eval = min(eval_size, MAX_EVAL_SAMPLES) if MAX_EVAL_SAMPLES is not None else eval_size
    effective_test = min(test_size, MAX_EVAL_SAMPLES) if MAX_EVAL_SAMPLES is not None else test_size
    train_batches = (effective_train + BATCH_SIZE - 1) // BATCH_SIZE
    eval_batches = (effective_eval + 47) // 48
    test_batches = (effective_test + 47) // 48
    total_train_steps = train_batches * EPOCHS
    intermediate_evaluations = total_train_steps // EVAL_EVERY_N_OPT_STEPS if EVAL_EVERY_N_OPT_STEPS else 0
    validation_passes = (2 if RUN_VALIDATION else 0) + intermediate_evaluations
    test_passes = 2 if RUN_TEST else 0
    print("Tamaños reales de los splits:", split_sizes)
    print(f"Lotes de entrenamiento: {train_batches} por época × {EPOCHS} = {total_train_steps}")
    print(f"Lotes por pasada de validación/test: {eval_batches}/{test_batches}")
    print(f"Pasadas estimadas: validación={validation_passes}, test={test_passes} (sin contar una posible pasada final si el flujo cambiase)")
    if result.stderr.strip():
        print("Mensajes del cargador del dataset:")
        print(result.stderr.strip())

Inspección omitida. Cambia INSPECT_DATASET a True para cargar y contar los splits.


## 4. Preparar el comando

El notebook pasa a Hydra las mismas opciones que la configuración del proyecto: modelo, idioma y par de idiomas; LoRA; límites de train/eval; épocas; evaluaciones; W&B y guardado. Se ejecuta `main.py` directamente para no ocultar esos parámetros detrás de `run_all.py`.

Con LoRA activado, `main.py` congela los pesos base y añade adaptadores. El cambio debería reflejarse en el conteo de parámetros entrenables del log. La referencia del algoritmo sigue siendo una copia del modelo, y las generaciones siguen usando el modelo NLLB completo.

In [5]:
def hydra_value(value):
    return "null" if value is None else str(value).lower() if isinstance(value, bool) else str(value)

dataset_config = f"eng_Latn-{LANG}"
RESULTS_ROOT = PROJECT_ROOT / "results" / "notebook_runs" / datetime.now().strftime("run_%Y%m%d_%H%M%S")
RUN_DIR = RESULTS_ROOT / "ours" / MODEL_KEY / LANG
overrides = [
    f"hydra.run.dir={RUN_DIR}",
    f"task.model.name={MODELS[MODEL_KEY]}",
    f"task.model.use_lora={hydra_value(USE_LORA)}",
    f"task.data.target_lang={LANG}",
    f"task.data.dataset_config_name={dataset_config}",
    f"task.data.max_train_samples={hydra_value(MAX_TRAIN_SAMPLES)}",
    f"task.data.max_eval_samples={hydra_value(MAX_EVAL_SAMPLES)}",
    f"task.training.epochs={EPOCHS}",
    f"task.training.batch_size={BATCH_SIZE}",
    f"task.training.use_wandb={hydra_value(USE_WANDB)}",
    f"task.training.save_model={hydra_value(SAVE_MODEL)}",
    f"task.eval.run={hydra_value(RUN_VALIDATION)}",
    f"task.eval.every_n_opt_steps={EVAL_EVERY_N_OPT_STEPS}",
    f"task.test.run={hydra_value(RUN_TEST)}",
]
COMMAND = ["uv", "run", "python", "main.py", *overrides]
print(f"Resultados: {RUN_DIR}")
print(shlex.join(COMMAND))

Resultados: /home/miguel/MT-via-Round-Trip-RL/results/notebook_runs/run_20261021_181345/ours/600M/rus_Cyrl
uv run python main.py hydra.run.dir=/home/miguel/MT-via-Round-Trip-RL/results/notebook_runs/run_20261021_181345/ours/600M/rus_Cyrl task.model.name=facebook/nllb-200-distilled-600M task.model.use_lora=true task.data.target_lang=rus_Cyrl task.data.dataset_config_name=eng_Latn-rus_Cyrl task.data.max_train_samples=16 task.data.max_eval_samples=32 task.training.epochs=2 task.training.batch_size=2 task.training.use_wandb=false task.training.save_model=false task.eval.run=true task.eval.every_n_opt_steps=0 task.test.run=true


## 5. Ejecutar

Antes de activar el interruptor, revisa el comando de la celda anterior. Ejecutar el entrenamiento descargará el modelo/dataset si no están en caché y consumirá GPU/CPU. El proceso se corta a los 15 minutos como máximo. Cada ejecución guarda sus métricas en una ruta nueva `results/notebook_runs/.../ours/600M/rus_Cyrl`, sin sobrescribir los resultados anteriores.

Con los límites iniciales, `main.py` verá como máximo 16 ejemplos de train y 32 de validación/test (8 lotes por época; 16 pasos de entrenamiento en total; una pasada de evaluación requiere un lote). La evaluación inicial/final de validación y test queda activa para obtener métricas antes/después; no hay evaluaciones intermedias. Para el recorrido más rápido, también puedes poner `RUN_VALIDATION = False`.

In [6]:
if not RUN_TRAINING:
    print("Entrenamiento omitido. Revisa ajustes/comando y cambia RUN_TRAINING a True para iniciarlo.")
else:
    subprocess.run(COMMAND, cwd=PROJECT_ROOT, check=True, timeout=840, env={**os.environ, "PYTHONIOENCODING": "utf-8"})

Entrenamiento omitido. Revisa ajustes/comando y cambia RUN_TRAINING a True para iniciarlo.


## 6. Mostrar resultados con el recolector del proyecto

Si el entrenamiento termina correctamente, la celda siguiente invoca `uv run python scripts/collect_results.py --results-dir <RESULTS_ROOT>`. Mostrará la misma tabla del proyecto, con chrF++ de test antes/después; al tratarse de solo 4 ejemplos, es un smoke test y no una puntuación publicable.

In [7]:
if not RUN_TRAINING:
    print("Recolector omitido: no se ha ejecutado un entrenamiento en esta sesión.")
else:
    collect_command = ["uv", "run", "python", "scripts/collect_results.py", "--results-dir", str(RESULTS_ROOT)]
    collected = subprocess.run(collect_command, cwd=PROJECT_ROOT, check=True, text=True, capture_output=True)
    print(collected.stdout)
    metrics_path = RUN_DIR / "metrics.jsonl"
    if not metrics_path.is_file() or '"split": "test"' not in metrics_path.read_text(encoding="utf-8"):
        raise RuntimeError(f"No encontré métricas de test para esta ejecución: {metrics_path}")

Recolector omitido: no se ha ejecutado un entrenamiento en esta sesión.


## Ideas para ajustar duración y alcance

- **Prueba ampliada seleccionada:** 16 ejemplos de train, 32 de eval/test, 2 épocas, LoRA, test activado y `EVAL_EVERY_N_OPT_STEPS=0`, con límite duro de 14 minutos.
- **Más entrenamiento:** aumenta `MAX_TRAIN_SAMPLES` o ponlo en `None`; recuerda que cada época recorre todo ese subconjunto.
- **Evaluar con menos coste:** limita `MAX_EVAL_SAMPLES` y aumenta `EVAL_EVERY_N_OPT_STEPS` (por ejemplo, 100) o déjalo en 0 para saltar las evaluaciones intermedias.
- **Fine-tuning completo:** cambia `USE_LORA=False`. Esto actualiza todos los pesos y requiere más memoria/cómputo; actívalo solo si esa es la comparación que necesitas.

Las muestras pequeñas sirven para validar que la tubería funciona, no para comparar calidad ni reportar resultados del experimento.